# Topic 5 — Matrix Product States

In Topic 4 we saw that the $n$-qubit wavefunction is a tensor with $n$ legs, that the singular value decomposition (SVD) across a cut gives the Schmidt decomposition, and that the states we care about have far fewer significant Schmidt values than the maximal $2^{n/2}$. This topic turns that observation into a data structure: the **matrix product state (MPS)**, in which the one big $n$-leg tensor is replaced by a chain of $n$ small tensors connected by **bonds** of dimension $\chi$. The memory cost is $O(n\chi^2)$ instead of $2^n$, and everything we did with state vectors — normalizing, measuring, applying gates, evolving in time — can be done directly on the chain.

1. **Matrix product states** — how to compress a state by successive SVDs, the MPS as an ansatz, examples, and `quimb`'s MPS class.
2. **Manipulations of MPS** — the gauge freedom of the representation and the **canonical form** with an orthogonality center, which makes norms, expectation values and correlation functions cheap.
3. **Time evolution with MPS** — applying one- and two-qubit gates with SVD truncation; quantum circuit simulation and the **time-evolving block decimation (TEBD)** algorithm; and the **time-dependent variational principle (TDVP)**, which evolves the MPS directly with the Hamiltonian.

**Setup.** As in Topic 4, run `uv sync` in the repository folder (then restart the kernel) so that `quimb` imports.

**Convention.** We follow `quimb` throughout this notebook: the sites of the chain are numbered $j = 0, 1, \ldots, n-1$ **from left to right**, a basis state is written $|\sigma_0 \sigma_1 \cdots \sigma_{n-1}\rangle$, and in a dense state vector (`psi.to_dense()`) site $0$ is the **most significant** bit. This is the opposite of the bit ordering of Topics 1–3 (qubit $0$ = least significant bit). All dense reference operators below are built with `qu.ikron(op, [2]*n, sites)`, which uses the same (site $0$ = most significant) ordering, so the two sides of every comparison agree.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import quimb as qu
import quimb.tensor as qtn
from scipy.linalg import expm, qr
from scipy.sparse.linalg import eigsh, expm_multiply
import warnings
warnings.filterwarnings('ignore', message='.*kahypar.*')      # an optional cotengra dependency we do not need

np.set_printoptions(precision=4, suppress=True, linewidth=120)

X, Z, I2 = qu.pauli('X'), qu.pauli('Z'), np.eye(2)

def ham_tfim(n, J, h, sparse=True):
    '''Dense/sparse transverse-field Ising Hamiltonian  H = -J sum Z_j Z_{j+1} - h sum X_j  (open chain),
    with site 0 as the most significant bit (the quimb convention).'''
    H = sum(-J * qu.ikron(Z, [2] * n, (j, j + 1), sparse=sparse) for j in range(n - 1))
    return H + sum(-h * qu.ikron(X, [2] * n, j, sparse=sparse) for j in range(n))

---

## 1. Matrix product states

### 1.1 Compression by successive SVDs

Write the state as a tensor with one leg per site,
$$
|\psi\rangle = \sum_{\sigma_0 \cdots \sigma_{n-1}} \psi_{\sigma_0 \sigma_1 \cdots \sigma_{n-1}}\, |\sigma_0 \sigma_1 \cdots \sigma_{n-1}\rangle ,
\qquad
\begin{array}{c} \sigma_0\ \ \sigma_1\ \cdots\ \sigma_{n-1} \\ |\quad\ \ |\qquad\quad\ \ | \\ \bigl[\qquad\quad \psi \qquad\quad\bigr] \end{array}
$$

**Step 1.** Group the legs as $(\sigma_0)\,|\,(\sigma_1 \cdots \sigma_{n-1})$: this is the $2 \times 2^{n-1}$ matrix of the Schmidt decomposition across the first bond. Apply the SVD and **truncate**, keeping at most $\chi$ singular values:
```
  σ0 σ1 σ2 ... σn-1          σ0          σ1 σ2 ... σn-1               σ0          σ1 σ2 ... σn-1
   |  |  |      |     SVD     |            |  |      |      truncate    |            |  |      |
  [      ψ        ]   ===   [U0]--(S0)--[     V0†      ]     ===>    [Ũ0]--(S̃0)--[     Ṽ0†      ]
```
Absorb the (truncated) singular values into the right factor, $\tilde V_0' := \tilde S_0 \tilde V_0^\dagger$, a tensor with legs $(\alpha_0, \sigma_1, \ldots, \sigma_{n-1})$ where $\alpha_0 = 1, \ldots, \chi_0$ is the new **bond index**.

**Step 2.** Regroup the legs of $\tilde V_0'$ as $(\alpha_0 \sigma_1)\,|\,(\sigma_2 \cdots \sigma_{n-1})$ — a $2\chi_0 \times 2^{n-2}$ matrix — SVD, truncate, absorb $\tilde S_1$ to the right:
```
      σ1 σ2 ... σn-1               σ1          σ2 ... σn-1
       |  |      |                  |           |      |
  α0--[     Ṽ0'     ]   ===>   α0--[Ũ1]--α1--[    Ṽ1'    ]
```
Repeating this $n-1$ times turns the one big box into a **chain**:
```
  σ0 σ1 σ2 ... σn-1          σ0     σ1     σ2         σn-1
   |  |  |      |             |      |      |           |
  [      ψ        ]    ≃     [Ũ0]--[Ũ1]--[Ũ2]-- ... --[Ũn-1]
                                α0     α1     α2    αn-2
```
Each $\tilde U_j$ has three legs, $(\alpha_{j-1}, \sigma_j, \alpha_j)$, i.e. it holds $\chi \times 2 \times \chi = 2\chi^2$ numbers.

**Matrices.** View $\tilde U_j$ as a pair of $\chi \times \chi$ matrices labelled by the physical index,
$$
\bigl[M_j^{[\sigma_j]}\bigr]_{\alpha_{j-1} \alpha_j} := [\tilde U_j]_{\alpha_{j-1}\, \sigma_j\, \alpha_j},
$$
except that $M_0^{[\sigma_0]}$ is a $1 \times \chi$ row and $M_{n-1}^{[\sigma_{n-1}]}$ a $\chi \times 1$ column. Then the compressed wavefunction is a product of matrices,
$$
|\tilde\psi\rangle = \sum_{\sigma_0 \cdots \sigma_{n-1}} M_0^{[\sigma_0]} M_1^{[\sigma_1]} \cdots M_{n-1}^{[\sigma_{n-1}]}\; |\sigma_0 \sigma_1 \cdots \sigma_{n-1}\rangle ,
$$
a **matrix product state**. The amplitude of any basis state is a product of $n$ small matrices.

- **Data usage** $\le 2\chi^2 n$: *linear* in $n$ for fixed $\chi$, compared with $2^n$. The number $\chi$ is the **bond dimension**.
- The bond dimension can **vary from bond to bond**, $\chi_j \le \min(2^{j+1}, 2^{n-1-j})$. Without truncation the chain is exact and $\chi_j$ equals the Schmidt rank of the cut $\{0..j\}\,|\,\{j+1..n-1\}$; the singular values of step $j$ are exactly the Schmidt values of that cut (the left factors $\tilde U_0 \cdots \tilde U_{j-1}$ have orthonormal columns, so they do not change singular values — this is the "canonical form" of Section 1.2).
- **Truncation error.** Discarding singular values $s_{\chi+1}, s_{\chi+2}, \ldots$ at one bond removes the weight $\epsilon = \sum_{i > \chi} s_i^2$ from the norm. The compressed state is a good approximation when the Schmidt values decay fast at every cut — true for ground states of local Hamiltonians, false for random states.

The cell below implements the algorithm in plain `numpy` and compares a random state with the ground state of the transverse-field Ising chain,
$$
H = -J \sum_{j=0}^{n-2} Z_j Z_{j+1} - h \sum_{j=0}^{n-1} X_j , \qquad J = 1,\ h = 1.2,\ n = 10 .
$$
We store the chain as a list of arrays `Ms[j]` of shape `(chi_left, 2, chi_right)`, with dummy bonds of dimension 1 at the two ends.

In [ ]:
def dense_to_mps(psi, chi_max=None, cutoff=1e-10):
    '''Compress a state vector into a list of 3-index arrays Ms[j][alpha_{j-1}, sigma_j, alpha_j] by successive SVDs.

    Keeps at most chi_max singular values per bond (all of them if None) and drops singular values below
    cutoff * s_max.  Also returns the singular values of every cut and the discarded weight per bond.'''
    psi = np.asarray(psi)
    n = int(np.round(np.log2(psi.size)))
    Ms, svals, discarded = [], [], []
    rest, chi = psi.reshape(1, -1), 1                 # rest[alpha_{j-1}, (sigma_j ... sigma_{n-1})]
    for j in range(n - 1):
        M = rest.reshape(chi * 2, -1)                 # group (alpha_{j-1} sigma_j) | (sigma_{j+1} ...)
        U, s, Vh = np.linalg.svd(M, full_matrices=False)
        keep = s > cutoff * s[0]
        if chi_max is not None:
            keep[chi_max:] = False
        k = int(np.sum(keep))
        Ms.append(U[:, :k].reshape(chi, 2, k))        # the new site tensor
        rest = s[:k, None] * Vh[:k]                   # absorb S into the right factor
        svals.append(s); discarded.append(float(np.sum(s[~keep] ** 2)))
        chi = k
    Ms.append(rest.reshape(chi, 2, 1))                # last site: (alpha_{n-2}, sigma_{n-1}, 1)
    return Ms, svals, discarded

def mps_to_dense(Ms):
    '''Contract the chain back into a state vector (exponential cost: only for checking!).'''
    T = Ms[0]
    for M in Ms[1:]:
        T = np.tensordot(T, M, axes=(-1, 0))          # join bond alpha_j
    return T.reshape(-1)

n, J, h = 10, 1.0, 1.2
rng = np.random.default_rng(0)
psi_rand = rng.normal(size=2 ** n) + 1j * rng.normal(size=2 ** n)
psi_rand /= np.linalg.norm(psi_rand)
E0, gs = eigsh(ham_tfim(n, J, h), k=1, which='SA')          # ground state (Lanczos, Topic 2)
gs = gs[:, 0]

for name, psi in [('random state', psi_rand), ('TFIM ground state', gs)]:
    Ms, svals, _ = dense_to_mps(psi)
    print(f"{name:18s}: bond dimensions {[M.shape[2] for M in Ms[:-1]]}, exact: {np.allclose(mps_to_dense(Ms), psi)}")
    print(f"{'':18s}  largest Schmidt values at the middle cut: {svals[n // 2 - 1][:6]}")

# truncation: fidelity |<psi|psi_chi>|^2 (after renormalizing psi_chi) versus bond dimension
chis = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32]
plt.figure(figsize=(5, 3.5))
for name, psi in [('random state', psi_rand), ('TFIM ground state', gs)]:
    infid, eps = [], []
    for chi in chis:
        Ms, _, discarded = dense_to_mps(psi, chi_max=chi)
        approx = mps_to_dense(Ms)
        infid.append(1 - abs(np.vdot(psi, approx)) ** 2 / np.vdot(approx, approx).real)
        eps.append(sum(discarded))
    plt.semilogy(chis, np.maximum(infid, 1e-16), 'o-', label=f'{name}: infidelity')
    plt.semilogy(chis, np.maximum(eps, 1e-16), 's--', alpha=0.6, label=f'{name}: total discarded weight')
plt.xlabel(r'bond dimension $\chi$'); plt.ylabel(r'$1 - |\langle\psi|\tilde\psi_\chi\rangle|^2$'); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.show()

The random state needs the full $\chi = 2^{n/2} = 32$ to be represented exactly (and compressing it to $\chi = 8$ loses most of the fidelity), whereas the ground state is reproduced to $10^{-6}$ with $\chi = 4$. Note also that the total discarded weight is a reliable estimate of the infidelity.

**The MPS as an ansatz.** Instead of *obtaining* an MPS from a state vector (which requires the $2^n$ amplitudes we are trying to avoid), we use the matrix-product form directly as an approximate **data structure / ansatz / variational wavefunction** for a quantum state:
$$
|\psi\rangle = \sum_{\sigma_0 \cdots \sigma_{n-1}} M_0^{[\sigma_0]} M_1^{[\sigma_1]} \cdots M_{n-1}^{[\sigma_{n-1}]}\; |\sigma_0 \cdots \sigma_{n-1}\rangle ,
\qquad
\begin{array}{c} \sigma_0\quad\ \ \sigma_1\quad\ \ \sigma_2\qquad\qquad\ \sigma_{n-1} \\ |\qquad\ \ |\qquad\ \ |\qquad\qquad\quad\ | \\ [M_0]\text{--}[M_1]\text{--}[M_2]\text{--}\cdots\text{--}[M_{n-1}] \end{array}
$$
The vertical legs $\sigma_j$ are the **physical** indices, the horizontal legs $\alpha_j$ the **virtual** (bond) indices. Every set of matrices defines a state; all algorithms of this topic update the matrices without ever forming the $2^n$-component vector.

*Examples.*
- $|0 \cdots 0\rangle$: $M_j^{[0]} = [1]$, $M_j^{[1]} = [0]$ ($1\times 1$ matrices, $\chi = 1$). Every product state $\otimes_j (a_j|0\rangle + b_j|1\rangle)$ is an MPS with $\chi = 1$: $M_j^{[0]} = [a_j]$, $M_j^{[1]} = [b_j]$.
- The GHZ state $\tfrac{1}{\sqrt 2}\bigl(|0\cdots0\rangle + |1\cdots1\rangle\bigr)$ needs $\chi = 2$:
$$
M_0^{[0]} = (1, 0),\quad M_0^{[1]} = (0, 1); \qquad
M_j^{[0]} = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix},\quad M_j^{[1]} = \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix}; \qquad
M_{n-1}^{[0]} = \tfrac{1}{\sqrt 2}\begin{pmatrix} 1 \\ 0 \end{pmatrix},\quad M_{n-1}^{[1]} = \tfrac{1}{\sqrt 2}\begin{pmatrix} 0 \\ 1 \end{pmatrix}.
$$
  The bond index acts as a *memory*: $\alpha = 0$ means "all sites so far were 0", $\alpha = 1$ "all were 1", and the matrices for site $j$ are nonzero only if $\sigma_j$ matches the memory. The product $M_0^{[\sigma_0]} \cdots M_{n-1}^{[\sigma_{n-1}]}$ is therefore $1/\sqrt 2$ for $\sigma = 0\cdots0$ and $1\cdots1$, and $0$ otherwise.

**MPS in `quimb`.** The class `qtn.MatrixProductState` holds the chain as a `TensorNetwork` of `Tensor`s (Topic 4), with the physical leg of site $j$ named `'k{j}'` and the tensor tagged `'I{j}'`; bond indices get automatically generated names.

| what | call |
|---|---|
| from a list of arrays | `qtn.MatrixProductState(arrays, shape='lpr')` — `shape` says which axis is the left bond, physical leg, right bond (the end arrays have only two axes: `'pr'` and `'lp'`) |
| product state | `qtn.MPS_product_state([v0, v1, ...])` (one length-2 vector per site), `qtn.MPS_computational_state('0110')` |
| named states | `qtn.MPS_ghz_state(n)`, `qtn.MPS_rand_state(n, bond_dim=chi)` |
| from a state vector | `qtn.MatrixProductState.from_dense(psi, dims=[2]*n, max_bond=chi, cutoff=1e-10)` (the algorithm of Section 1.1) |
| back to a state vector | `psi.to_dense()` (site 0 = most significant bit), one amplitude: `psi.amplitude('0110')` |
| inspect | `psi.L`, `psi[j]` (the `Tensor` of site $j$), `psi.site_ind(j)`, `psi.bond(j, j+1)`, `psi.bond_sizes()`, `psi.arrays`, `psi.draw()` |
| norm | `psi.norm()`, or `psi.H @ psi` $= \langle\psi|\psi\rangle$ (contracts the chain with its conjugate — efficiently, Section 1.2) |

The cell below builds the GHZ state by hand, compares it with the built-in constructors, and shows the data layout.

In [ ]:
n = 5
# GHZ by hand, arrays in 'lpr' = (left bond, physical, right bond) layout
M0 = np.zeros((2, 2));    M0[0, 0] = 1;    M0[1, 1] = 1            # (sigma_0, alpha_0)        : 'pr'
Mj = np.zeros((2, 2, 2)); Mj[0, 0, 0] = 1; Mj[1, 1, 1] = 1         # (alpha_{j-1}, sigma_j, alpha_j)
Mn = np.zeros((2, 2));    Mn[0, 0] = 1;    Mn[1, 1] = 1            # (alpha_{n-2}, sigma_{n-1}) : 'lp'
ghz = qtn.MatrixProductState([M0] + [Mj] * (n - 2) + [Mn / np.sqrt(2)], shape='lpr')
print(ghz)

print("bond dimensions          :", ghz.bond_sizes())
print("site 2 tensor            :", ghz[2].inds, ghz[2].shape, " physical index:", ghz.site_ind(2), " bond (2,3):", ghz.bond(2, 3))
print("amplitudes of 00000, 11111, 01000:", ghz.amplitude('00000'), ghz.amplitude('11111'), ghz.amplitude('01000'))
print("norm^2  <psi|psi>        :", ghz.H @ ghz)
print("dense == built-in GHZ    :", np.allclose(ghz.to_dense(), qtn.MPS_ghz_state(n).to_dense()))
print("dense == (|0..0>+|1..1>)/√2:", np.allclose(np.asarray(ghz.to_dense()).ravel(), (qu.computational_state('0' * n) + qu.computational_state('1' * n)).ravel() / np.sqrt(2)))
ghz.draw(color=[f'I{j}' for j in range(n)], show_inds='bond-size', figsize=(6, 2))

# other constructors
p = qtn.MPS_computational_state('01101')
print("computational state 01101: bonds", p.bond_sizes(), " nonzero dense index:", np.flatnonzero(np.asarray(p.to_dense()).ravel()), "= 0b01101 =", 0b01101)
plus = qtn.MPS_product_state([np.array([1, 1]) / np.sqrt(2)] * n)
print("|+++++>                  : bonds", plus.bond_sizes(), " all amplitudes equal:", np.allclose(np.asarray(plus.to_dense()).ravel(), 2 ** (-n / 2)))

def domain_wall_mps(n):
    '''|+...+ -...- +...+> with |-> on the middle third of the chain: a chi = 1 product state (used in Section 2).'''
    plus, minus = np.array([1, 1]) / np.sqrt(2), np.array([1, -1]) / np.sqrt(2)
    return qtn.MPS_product_state([(minus if n // 3 <= j < 2 * n // 3 else plus).astype(complex) for j in range(n)])

dw = domain_wall_mps(6)
v = np.asarray(dw.to_dense()).ravel()
print("domain wall |++--++>     : bonds", dw.bond_sizes(), " <X_j> =", np.array([np.vdot(v, qu.ikron(X, [2] * 6, j) @ v).real for j in range(6)]))

# a random MPS with 100 sites and chi = 16: a 2^100-dimensional vector described by ~ 50 000 numbers
big = qtn.MPS_rand_state(100, bond_dim=16, seed=0)
print("n=100, chi=16            : parameters =", sum(t.size for t in big.tensors), " norm^2 =", big.H @ big)

### Exercise 1: building matrix product states

**1.** *Cluster state.* The one-dimensional cluster state (the resource state of measurement-based quantum computing) is prepared by putting every qubit in $|+\rangle$ and applying a controlled-Z gate to every pair of neighbours,
$$
|C\rangle = \prod_{j=0}^{n-2} \mathrm{CZ}_{j,j+1}\,|{+}\rangle^{\otimes n}, \qquad \mathrm{CZ}\,|\sigma\sigma'\rangle = (-1)^{\sigma\sigma'}\,|\sigma\sigma'\rangle .
$$
It is an MPS with $\chi = 2$ and the matrices
$$
M_0^{[0]} = \tfrac{1}{\sqrt 2}(1, 0),\quad M_0^{[1]} = \tfrac{1}{\sqrt 2}(0, 1); \qquad
M_j^{[0]} = \tfrac{1}{\sqrt 2}\begin{pmatrix} 1 & 0 \\ 1 & 0 \end{pmatrix},\quad M_j^{[1]} = \tfrac{1}{\sqrt 2}\begin{pmatrix} 0 & 1 \\ 0 & -1 \end{pmatrix}; \qquad
M_{n-1}^{[0]} = \tfrac{1}{\sqrt 2}\begin{pmatrix} 1 \\ 1 \end{pmatrix},\quad M_{n-1}^{[1]} = \tfrac{1}{\sqrt 2}\begin{pmatrix} 1 \\ -1 \end{pmatrix}.
$$
The bond index remembers the value of the previous qubit, and each site contributes the sign $(-1)^{\sigma_{j-1}\sigma_j}$ of the CZ gate with its left neighbour and a factor $1/\sqrt 2$.

*(a) Build it.* Write `cluster_state_mps(n)` that returns the state as a `qtn.MatrixProductState`, built like the GHZ cell above: a first array of shape `(2, 2)` with axes $(\sigma_0, \alpha_0)$, then $n - 2$ copies of a bulk array of shape `(2, 2, 2)` with axes $(\alpha_{j-1}, \sigma_j, \alpha_j)$, then a last array of shape `(2, 2)` with axes $(\alpha_{n-2}, \sigma_{n-1})$, all passed with `shape='lpr'`. Watch the index order: the physical index sits in the middle, so the bulk array element `M[alpha, sigma, beta]` is $[M_j^{[\sigma]}]_{\alpha\beta}$. For $n = 5$, check it against the dense vector built from the definition: start from $|{+}\rangle^{\otimes n}$ (`np.ones(2**n) / 2**(n / 2)`) and apply the $n - 1$ gates $\mathrm{CZ}_{j,j+1}$ as $2^n \times 2^n$ matrices `qu.pkron(np.diag([1, 1, 1, -1]), [2] * n, [j, j + 1])` (`qu.pkron` is the version of `qu.ikron` for an operator acting on several sites); compare with `np.allclose`, flattening both vectors with `np.asarray(...).ravel()` as in the GHZ check. For $n = 50$, check that `psi.H @ psi` $= 1$.

*(b) Correlation functions.* For $n = 50$ the dense vector would have $2^{50} \approx 10^{15}$ entries, but the expectation value of a product of single-site operators is a cheap contraction of the MPS: applying a $2 \times 2$ operator $O$ to site $j$ only changes the tensor of that site, which `phi.gate_(O, j, contract=True)` does in place. Write `expect_ops(psi, ops)` that returns $\langle\psi|\prod_j O_j|\psi\rangle$ for a dictionary `ops = {j: O_j}`, by applying the operators to a copy `phi = psi.copy()` and returning `psi.H @ phi`. Then, for $n = 50$, compute
- the two-point functions $\langle Z_{20} Z_j\rangle$ and $\langle X_{20} X_j\rangle$ for $j = 21, \ldots, 30$;
- the **stabilizers** $\langle Z_{j-1} X_j Z_{j+1}\rangle$ for all $0 < j < n-1$, and $\langle X_0 Z_1\rangle$, $\langle Z_{n-2} X_{n-1}\rangle$ at the two ends;
- the **string correlator** $\langle Z_i\, X_{i+1} X_{i+3} \cdots X_{j-1}\, Z_j\rangle$ (an $X$ on every second site between the two $Z$s) for $i = 10$ and $j = 12, 14, \ldots, 40$.

Explain the results. (Hint: use $X|{+}\rangle = |{+}\rangle$ and $\mathrm{CZ}_{12}\, X_2\, \mathrm{CZ}_{12} = Z_1 X_2$ to show that $K_j|C\rangle = |C\rangle$ for $K_j = Z_{j-1} X_j Z_{j+1}$. What is the product $K_{i+1} K_{i+3} \cdots K_{j-1}$? And if an operator anticommutes with one of the $K_j$, what is its expectation value?) All two-point correlations vanish, yet the string correlator does not decay with distance: the cluster state has hidden long-range *string order*, the hallmark of a symmetry-protected topological phase.

**2.** *Product states, two ways.* Build the alternating product state $|{+}{-}{+}{-}{+}{-}\rangle$ ($n = 6$) with `qtn.MPS_product_state`, and build the same state with `qtn.MatrixProductState(arrays, shape='lpr')` from arrays of shape `(2, 1)`, `(1, 2, 1)`, …, `(1, 2)`. Check that both have bond dimension 1, that `psi.H @ psi` $= 1$, that their dense vectors agree, and compute $\langle X_j\rangle$ from the dense vector as in the demo.

In [ ]:
# --- Part 1: cluster state ---
def cluster_state_mps(n):
    pass


In [ ]:
# --- Part 2: product states, two ways ---


---

### 1.2 Manipulations of MPS

#### 1.2.1 Gauge degrees of freedom

The matrices of an MPS are **not unique**. Insert $\mathbb{1} = X_j X_j^{-1}$ on every bond, with any invertible $\chi_j \times \chi_j$ matrix $X_j$:
```
  [M0]--[M1]--[M2]-- ... --[Mn-1]   =   [M0]-[X0]-[X0⁻¹]-[M1]-[X1]-[X1⁻¹]-[M2]- ... -[Xn-2⁻¹]-[Mn-1]

                                    =   [N0]--[N1]--[N2]-- ... --[Nn-1]
```
where
$$
N_0^{[\sigma]} = M_0^{[\sigma]} X_0, \qquad N_j^{[\sigma]} = X_{j-1}^{-1} M_j^{[\sigma]} X_j, \qquad N_{n-1}^{[\sigma]} = X_{n-2}^{-1} M_{n-1}^{[\sigma]} .
$$
The new matrices $N_j$ describe exactly the same state. This **gauge freedom** is a nuisance (two different-looking MPS can be equal) and an opportunity: we can *choose* the gauge that makes computations easy. The cell below checks the statement numerically.

In [ ]:
n = 6
psi = qtn.MPS_rand_state(n, bond_dim=3, seed=2, dtype='complex128')
rng = np.random.default_rng(3)

# insert X_j X_j^{-1} on every bond, working on the 'lpr' arrays of the chain
Ms = [psi[j].transpose(*([psi.bond(j - 1, j)] if j > 0 else []), psi.site_ind(j), *([psi.bond(j, j + 1)] if j < n - 1 else [])).data for j in range(n)]
Ns = [M.copy() for M in Ms]
for j in range(n - 1):
    chi = Ms[j].shape[-1]
    Xj = rng.normal(size=(chi, chi)) + 1j * rng.normal(size=(chi, chi))      # random invertible matrix
    Ns[j] = Ns[j] @ Xj                                                        # M_j X_j         (last axis = bond j)
    Ns[j + 1] = np.tensordot(np.linalg.inv(Xj), Ns[j + 1], axes=(1, 0))      # X_j^{-1} M_{j+1} (first axis = bond j)
phi = qtn.MatrixProductState(Ns, shape='lpr')

print("tensors changed      :", not np.allclose(phi[2].data, psi[2].data), " (site 2 norms:", round(float(psi[2].norm()), 3), "vs", round(float(phi[2].norm()), 3), ")")
print("same state           :", np.allclose(phi.to_dense(), psi.to_dense()))

#### 1.2.2 Canonical form

**Left-canonical tensors.** Given a site tensor $M$, we want a gauge transformation $X$ such that $N = MX$ satisfies
$$
\sum_{\alpha, \sigma} [N]^*_{\alpha \sigma \beta}\, [N]_{\alpha \sigma \beta'} = \delta_{\beta \beta'}
\quad\Longleftrightarrow\quad
\sum_\sigma N^{[\sigma]\dagger} N^{[\sigma]} = \mathbb{1},
\qquad
\begin{array}{c} \text{---}[N^*]\text{---} \\ |\quad\ \ \ \\ \text{---}[N]\text{---} \end{array}
\ \text{(left and physical legs joined)} \ =\ \text{---}\!\!\mathbb{1}\!\!\text{---}
$$
i.e. $N$, viewed as the $(\chi_{\rm l}\cdot 2) \times \chi_{\rm r}$ matrix $N_{(\alpha\sigma), \beta}$, has **orthonormal columns** (it is an *isometry*). Such a tensor is called **left-canonical** and we denote it by $A$.

**QR decomposition.** Any matrix can be written as $M = QR$ with $Q^\dagger Q = \mathbb{1}$ and $R$ upper triangular (`numpy.linalg.qr` / `scipy.linalg.qr(..., mode='economic')`). So:
```
  --[M]--   =   --[Q]--[R]--        where   [Q*]
                                            |      =  --1--     and  R is absorbed into the tensor on the right.
                                            [Q ]
```
Doing this iteratively from left to right — QR of $M_0$, absorb $R$ into $M_1$, QR of the result, absorb into $M_2$, ... — makes every tensor except the last one left-canonical. In `quimb`: `psi.left_canonize()`.
```
  [A0]--[A1]--[A2]-- ... --[An-2]--[Mn-1]        every A_j satisfies  sum_σ A†A = 1
```
The last tensor then carries the whole norm, $\langle\psi|\psi\rangle = \sum_\sigma \|M_{n-1}^{[\sigma]}\|^2$ (contract the chain with its conjugate from the left: every $A_j^\dagger A_j$ pair collapses to the identity).

**Right-canonical tensors** $B$ are defined by the mirror condition $\sum_\sigma B^{[\sigma]} B^{[\sigma]\dagger} = \mathbb{1}$ (orthonormal *rows* of $B_{\alpha, (\sigma\beta)}$), obtained by sweeping from right to left: `psi.right_canonize()`.

**Mixed canonical form.** Doing it from both sides up to a chosen site $j$ gives
```
  [A0]--[A1]-- ... --[Aj-1]--[Cj]--[Bj+1]-- ... --[Bn-1]
                               ^
                   orthogonality center (OC)
```
with all $A$'s left-canonical and all $B$'s right-canonical; $C_j$ is the **orthogonality center**. In `quimb`: `psi.canonize(j)`; `psi.calc_current_orthog_center()` finds it, and `psi.shift_orthogonality_center(j, k)` moves it with $|j - k|$ QR steps.

**Why it matters: cheap contractions.** With the OC at site $j$, everything to the left and to the right of it collapses to the identity when the chain is contracted with its conjugate:

- *Norm.* $\langle\psi|\psi\rangle = \sum_{\alpha\sigma\beta} |[C_j]_{\alpha\sigma\beta}|^2 = \|C_j\|^2$: a single tensor, $O(\chi^2)$.
- *Local expectation value.* $\langle\psi| O_j |\psi\rangle = \sum C_j^*\, O\, C_j$ (contract $O$ into the physical leg of $C_j$, then with $C_j^*$): $O(\chi^2)$, independent of $n$ — compared with $2^n$ for a state vector.
- *Correlation function* $\langle\psi| O_i P_j |\psi\rangle$, $i < j$: put the OC at $i$, then contract site by site from $i$ to $j$. Each intermediate step multiplies by the **transfer matrix** $E_m = \sum_\sigma B_m^{[\sigma]*} \otimes B_m^{[\sigma]}$ of site $m$, and the sites beyond $j$ collapse to the identity. Cost $O((j-i)\,\chi^3)$.
- *Schmidt values.* The SVD of the OC viewed as the matrix $[C_j]_{(\alpha\sigma),\beta} = U S V^\dagger$ gives the Schmidt values of the cut between sites $j$ and $j+1$, because $A_0 \cdots A_{j-1} U$ and $V^\dagger B_{j+1} \cdots B_{n-1}$ are both isometries. The entanglement entropy of any cut therefore costs $O(\chi^3)$: `psi.schmidt_values(i)` returns the *squared* Schmidt values $s^2$ of the cut between sites $i-1$ and $i$ (left block of $i$ sites), and `psi.entropy(i)` the corresponding entropy — in **bits** (log base 2); we use natural logarithms as in Topic 4.

```
            [A0*]--[A1*]-- ... --[Cj*]-- ... --[Bn-1*]        [Cj*]
  <ψ|ψ> =     |      |             |              |      =      |   |      ,
            [A0 ]--[A1 ]-- ... --[Cj ]-- ... --[Bn-1 ]        [Cj ]

                                                               [Cj*]
  <ψ|Zj|ψ> =  (same, with Z on the physical leg of site j) =    [Z ]
                                                               [Cj ]
```
This is also why truncating an MPS with the SVD is *optimal* only in canonical form: the singular values are the true Schmidt values only if the rest of the chain is isometric (Section 2.1).

The cells below demonstrate the canonical forms in `quimb` and then use the OC to evaluate the norm, local expectation values and a correlation function, checking each against the dense state vector.

In [ ]:
n = 8
psi = qtn.MPS_rand_state(n, bond_dim=4, seed=1, dtype='complex128')

def is_left_isometry(psi, j):
    '''Check  sum_{alpha sigma} A*_{alpha sigma beta} A_{alpha sigma beta'} = delta  for the tensor of site j.'''
    t, rb = psi[j], psi.bond(j, j + 1)
    E = t.H.reindex({rb: 'beta_prime'}) @ t          # contract left bond and physical leg; keep the right bond open
    return np.allclose(E.data, np.eye(t.ind_size(rb)))

def is_right_isometry(psi, j):
    t, lb = psi[j], psi.bond(j - 1, j)
    E = t.H.reindex({lb: 'alpha_prime'}) @ t
    return np.allclose(E.data, np.eye(t.ind_size(lb)))

print("random MPS   : site 2 left-isometric?", is_left_isometry(psi, 2), "  OC region:", psi.calc_current_orthog_center())

left = psi.copy(); left.left_canonize()                 # in place
print("left-canonize: sites 0..6 left-isometric?", [is_left_isometry(left, j) for j in range(n - 1)],
      " OC:", left.calc_current_orthog_center(), " norm^2 from the last tensor:", round(float(left[n - 1].norm() ** 2), 6), "= <psi|psi> =", round(float((psi.H @ psi).real), 6))

right = psi.copy(); right.right_canonize()
print("right-canonize: sites 1..7 right-isometric?", [is_right_isometry(right, j) for j in range(1, n)], " OC:", right.calc_current_orthog_center())

mixed = psi.copy(); mixed.canonize(3)                    # mixed canonical form with the OC at site 3
print("canonize(3)  : left-iso", [is_left_isometry(mixed, j) for j in range(3)], " right-iso", [is_right_isometry(mixed, j) for j in range(4, n)],
      " OC:", mixed.calc_current_orthog_center())
mixed.shift_orthogonality_center(3, 5)                   # 2 QR steps
print("after shift  : OC:", mixed.calc_current_orthog_center(), " still the same state:", np.allclose(mixed.to_dense(), psi.to_dense()))
mixed.draw(color=['I5'], show_inds='bond-size', figsize=(7, 2))   # the OC highlighted

In [ ]:
def expect_local(psi, G, j):
    '''<psi|G_j|psi> / <psi|psi> using the orthogonality center.'''
    psi = psi.copy(); psi.canonize(j)
    C, k = psi[j], psi.site_ind(j)
    GC = (qtn.Tensor(G, inds=('out', k)) @ C).reindex({'out': k})     # G acting on the physical leg of C
    return (C.H @ GC) / (C.H @ C)

def local_expectations(psi, G):
    '''<G_j> for every site j, moving the OC one site at a time (O(n chi^3) in total).'''
    psi = psi.copy(); psi.canonize(0)
    out = []
    for j in range(psi.L):
        if j > 0:
            psi.shift_orthogonality_center(j - 1, j)
        C, k = psi[j], psi.site_ind(j)
        GC = (qtn.Tensor(G, inds=('out', k)) @ C).reindex({'out': k})
        out.append(((C.H @ GC) / (C.H @ C)).real)
    return np.array(out)

def correlation_oc(psi, A, i, B, j):
    '''<psi|A_i B_j|psi> / <psi|psi> for i < j: OC at i, then transfer matrices from i to j.'''
    psi = psi.copy(); psi.canonize(i)
    nrm = psi[i].H @ psi[i]
    t, k = psi[i], psi.site_ind(i)
    At = (qtn.Tensor(A, inds=('out', k)) @ t).reindex({'out': k})
    E = t.H.reindex({psi.bond(i, i + 1): 'up'}) @ At                   # open legs: ('up' = bra bond, ket bond)
    for m in range(i + 1, j):                                          # transfer matrices of the sites in between
        t = psi[m]
        E = (E @ t.H.reindex({psi.bond(m - 1, m): 'up', psi.bond(m, m + 1): 'up2'}) @ t).reindex({'up2': 'up'})
    t, k = psi[j], psi.site_ind(j)
    Bt = (qtn.Tensor(B, inds=('out', k)) @ t).reindex({'out': k})
    return (E @ t.H.reindex({psi.bond(j - 1, j): 'up'}) @ Bt) / nrm     # sites > j collapse to the identity

def entropy_cut(psi, i):
    '''Entanglement entropy (natural log) of the cut between sites i-1 and i, from the Schmidt values of the OC.'''
    p = np.asarray(psi.schmidt_values(i)); p = p[p > 1e-15]
    return float(-np.sum(p * np.log(p)))

n = 10
psi_gs = qtn.MatrixProductState.from_dense(gs, dims=[2] * n, cutoff=1e-15)   # the TFIM ground state as an MPS (Section 1.1), no truncation
Xd = lambda j: qu.ikron(X, [2] * n, j); Zd = lambda j: qu.ikron(Z, [2] * n, j)

print("norm^2 from the OC   :", psi_gs.copy().canonize(4)[4].norm() ** 2, "  psi.H @ psi:", psi_gs.H @ psi_gs)
print("<X_3>  OC            :", expect_local(psi_gs, X, 3), "  dense:", np.vdot(gs, Xd(3) @ gs).real, "  quimb:", psi_gs.local_expectation_canonical(X, 3))
print("<X_j>  profile       :", local_expectations(psi_gs, X))
print("<Z_2 Z_6>  OC        :", correlation_oc(psi_gs, Z, 2, Z, 6), "  dense:", np.vdot(gs, Zd(2) @ Zd(6) @ gs).real, "  quimb:", psi_gs.local_expectation_canonical(Z & Z, (2, 6)))
print("Schmidt values cut 5 :", psi_gs.schmidt_values(5), "  dense SVD:", np.linalg.svd(gs.reshape(2 ** 5, 2 ** 5), compute_uv=False)[:4] ** 2, " (quimb returns s^2)")
print("entropy of cut 5     :", entropy_cut(psi_gs, 5), "  quimb's psi.entropy(5) in bits:", psi_gs.entropy(5), "=", psi_gs.entropy(5) * np.log(2), "nats")

(Both `quimb` helpers used above, `local_expectation_canonical` and `schmidt_values`, do exactly what our hand-written functions do: move the OC and contract a few tensors.)

### Exercise 2: expectation values from the orthogonality center

Use the functions of the demo (`is_left_isometry`, `is_right_isometry`, `expect_local`, `local_expectations`, `correlation_oc`, `entropy_cut`) together with the `quimb` built-ins.

**1.** *GHZ state.* Take `psi = qtn.MPS_ghz_state(8)`, put the OC on site 4 with `canonize`, and check with `is_left_isometry` / `is_right_isometry` which tensors are isometric. Read off the norm from the OC tensor alone and compare with `psi.H @ psi`. Then compute $\langle Z_j\rangle$ for all $j$, $\langle Z_0 Z_r\rangle$ for $r = 1, \ldots, 7$, and the entanglement entropy of every cut; compare with `local_expectation_canonical` and `schmidt_values`, and explain the three results ($0$, $1$, $\ln 2$) from the form of the state.

**2.** *Cluster state.* Repeat part 1 for the cluster state of Exercise 1 with $n = 8$ (which is small enough to check everything against the dense vector, `qu.ikron`). Explain the values of $\langle Z_j\rangle$, $\langle Z_0 Z_r\rangle$ and of the entropies. (Hint: the CZ gates only change signs, so all amplitudes have the same magnitude; and only the gate $\mathrm{CZ}_{k-1,k} = |0\rangle\langle 0|_{k-1} + |1\rangle\langle 1|_{k-1}\, Z_k$ acts across the cut after $k$ sites.) The cluster state is as entangled as the GHZ state across every cut, yet it has no $ZZ$ correlations: its correlations are the stabilizers and the string order of Exercise 1.

In [ ]:
# --- Part 1: GHZ state ---


In [ ]:
# --- Part 2: cluster state ---


---

## 2. Time evolution with MPS

### 2.1 Applying gates to an MPS

**Single-qubit gate** $U$ on site $j$. Move the OC to site $j$ and contract $U$ into the physical leg:
```
    [U]
     |        =    [Cj']          C'_j^{[σ']} = Σ_σ U_{σ'σ} C_j^{[σ]}
  --[Cj]--       --[   ]--
```
Nothing else changes, and $C'_j$ is still the OC. Cost $O(\chi^2)$. (In fact a unitary on the physical leg preserves the isometry conditions, so it may be applied to *any* site tensor; moving the OC is only needed when we want to truncate afterwards.)

**Two-qubit gate** $U$ on neighbouring sites $(j, j+1)$. Move the OC to site $j$ (the left one), contract the gate with $C_j$ and $B_{j+1}$ into a two-site tensor $\Theta$, and split it again with an SVD:
```
    [    U    ]                                              SVD                        truncate
     |      |        =   --[    Θ    ]--   = (reshape) --[ Θ ]--    ===   --[U]--(S)--[V†]--   ===>   --[Ũ]--(S̃)--[Ṽ†]--
  --[Cj]--[Bj+1]--          |      |                   (2χ)x(2χ)            |       |                   |        |

                     =   --[Cj']--[B'j+1]--      with  C'_j = Ũ  (a left isometry → the new A_j)  and  B'_{j+1} = S̃ Ṽ†  (the new OC).
```
The grouping is $(\alpha_{j-1}\, \sigma_j')\,|\,(\sigma_{j+1}'\, \alpha_{j+1})$, i.e. a $2\chi_{j-1} \times 2\chi_{j+1}$ matrix. After the SVD the new bond has up to $\min(2\chi_{j-1}, 2\chi_{j+1})$ singular values: **every entangling gate can double the bond dimension**, which is why we **truncate** to the largest $\chi_{\max}$ singular values (and/or drop those below a `cutoff`), discarding the weight $\epsilon = \sum_{i > \chi_{\max}} s_i^2$. Because $C_j B_{j+1}$ sits in mixed canonical form (the rest of the chain is isometric), the singular values are the true Schmidt values of this cut and the truncation is the *optimal* rank-$\chi_{\max}$ approximation of the state (Topic 4). The OC ends up on site $j+1$, convenient when the next gate acts further to the right. Cost: $O(\chi^3)$ for the SVD.

Gates on **non-adjacent** sites are handled by inserting SWAP gates until the sites are neighbours (each SWAP is itself a two-site gate, with its own truncation).

In `quimb` this is a one-liner. For a two-site gate `G` (a $4\times 4$ matrix, sites read left to right) on sites `(j, j+1)`:
```python
psi.gate_(G, (j, j + 1), contract='swap+split', max_bond=chi, cutoff=1e-10, info=info)
```
moves the OC to the pair (swapping first if the sites are not adjacent), contracts, splits with the truncated SVD, and leaves the OC on the right site. The dictionary `info = {'cur_orthog': ...}` lets `quimb` remember where the OC is between calls, so that it does not have to recompute the canonical form for every gate (`'calc'` means "work it out"). A single-site gate is `psi.gate_(G, j, contract=True, info=info)`. The trailing underscore means *in place*; `psi.gate(...)` returns a modified copy.

The cell below performs the two-site step by hand with `quimb` tensors and then with `gate_`, and shows the growth of the bond dimension under repeated gates.

In [ ]:
n = 8
psi = qtn.MPS_ghz_state(n)                                           # chi = 2 everywhere
G = expm(1j * 0.4 * np.kron(X, X))                                   # e^{+i 0.4 X X}: an entangling two-site gate (GHZ is not an eigenstate of XX)
j = 3

# --- by hand: OC to site j, contract gate with C_j B_{j+1}, split with a truncated SVD ---
phi = psi.copy(); phi.canonize(j)
kj, kj1 = phi.site_ind(j), phi.site_ind(j + 1)
tG = qtn.Tensor(G.reshape(2, 2, 2, 2), inds=('oj', 'oj1', kj, kj1))                  # (out_j, out_j+1, in_j, in_j+1)
theta = (tG @ phi[j] @ phi[j + 1]).reindex({'oj': kj, 'oj1': kj1})                   # the two-site tensor Theta
print("Theta legs:", theta.inds, theta.shape)
left_legs = [phi.bond(j - 1, j), kj]                                                 # group (alpha_{j-1} sigma_j) | (sigma_{j+1} alpha_{j+1})
s = theta.split(left_inds=left_legs, method='svd', get='values')
print("singular values of Theta:", s, "-> the bond can grow from 2 to", np.sum(s > 1e-10))
tl, tr = theta.split(left_inds=left_legs, method='svd', absorb='right', max_bond=4, cutoff=1e-10, bond_ind=phi.bond(j, j + 1))
phi[j].modify(data=tl.transpose(*phi[j].inds).data)        # put the two pieces back into the chain
phi[j + 1].modify(data=tr.transpose(*phi[j + 1].inds).data)
print("by hand   : bonds", phi.bond_sizes(), " OC:", phi.calc_current_orthog_center(),
      " correct:", np.allclose(np.asarray(phi.to_dense()).ravel(), qu.ikron(G, [2] * n, (j, j + 1)) @ np.asarray(psi.to_dense()).ravel()))

# --- with quimb's gate_ ---
chi = psi.copy(); info = {'cur_orthog': 'calc'}
chi.gate_(G, (j, j + 1), contract='swap+split', max_bond=4, cutoff=1e-10, info=info)
print("gate_     : bonds", chi.bond_sizes(), " OC:", info['cur_orthog'],
      " correct:", np.allclose(np.asarray(chi.to_dense()).ravel(), qu.ikron(G, [2] * n, (j, j + 1)) @ np.asarray(psi.to_dense()).ravel()))
chi.gate_(X, 2, contract=True, info=info)                   # a single-site gate
print("single-site gate_: bonds", chi.bond_sizes(), " OC:", info['cur_orthog'])

# --- bond growth under a brickwork of random two-site unitaries, with and without truncation ---
rng = np.random.default_rng(0)
for max_bond in [None, 8]:
    phi = qtn.MPS_computational_state('0' * n); info = {'cur_orthog': 'calc'}
    growth = [max(phi.bond_sizes())]
    for layer in range(6):
        for jj in range(layer % 2, n - 1, 2):
            phi.gate_(qu.rand_uni(4, seed=rng.integers(1 << 30)), (jj, jj + 1), contract='swap+split', max_bond=max_bond, cutoff=1e-10, info=info)
        growth.append(max(phi.bond_sizes()))
    print(f"max bond after each layer, max_bond={max_bond}: {growth}   (norm^2 after truncation: {float((phi.H @ phi).real):.4f})")

### 2.2 Quantum circuit simulation and TEBD

Since any quantum circuit is a sequence of one- and two-qubit gates, the two operations of Section 2.1 give a complete **MPS circuit simulator**: start from $|0\cdots0\rangle$ ($\chi = 1$) and apply the gates one by one, truncating the bond dimension after each two-qubit gate. `quimb` provides this as `qtn.CircuitMPS`, with the same interface as the `Circuit` class of Topic 4 (`circ.h(i)`, `circ.cnot(i, j)`, `circ.rzz(theta, i, j)`, `circ.amplitude(b)`, `circ.local_expectation(G, where)`, `circ.sample(C)`, ...) but with the state stored as an MPS, `circ.psi`:
```python
circ = qtn.CircuitMPS(N=n, max_bond=chi, cutoff=1e-10)
```
The result is exact as long as no truncation happens, and otherwise it is an approximation whose quality depends on how much entanglement the circuit generates.

**Time evolution.** To compute $e^{-iHt}|\psi\rangle$ for a Hamiltonian with nearest-neighbour terms, $H = \sum_j h_{j,j+1}$, use the **Trotter–Suzuki decomposition** of Topic 3 to turn the propagator into a circuit:
$$
e^{-iH\,\delta t} \approx \prod_{j} e^{-i h_{j,j+1}\,\delta t} \quad (\text{first order}),
\qquad
e^{-iHt} \approx \bigl(U_{\rm even}(\delta t/2)\, U_{\rm odd}(\delta t)\, U_{\rm even}(\delta t/2)\bigr)^{t/\delta t} \quad (\text{second order}),
$$
where $U_{\rm even/odd}$ is the product of the commuting two-site gates $e^{-i h_{j,j+1}\delta t}$ on even/odd bonds. Applying these gates one by one with the SVD-truncation step is the **time-evolving block decimation (TEBD)** algorithm (Vidal 2003). It has two sources of error: the **Trotter error** ($O(\delta t)$ per unit time for first order, $O(\delta t^2)$ for second order) and the **truncation error** (the discarded weight at every SVD, controlled by $\chi_{\max}$).

For the transverse-field Ising model it is convenient to keep the field terms as separate single-site gates, as in Topic 4: one first-order step is
$$
U(\delta t) = \prod_{j=0}^{n-1} e^{+i h\,\delta t\, X_j} \;\prod_{j=0}^{n-2} e^{+i J\,\delta t\, Z_j Z_{j+1}} ,
$$
applied right factor first.

**Entanglement growth.** After a *global quench* (a product state evolved with a generic Hamiltonian) the entanglement entropy across a cut grows linearly in time, $S(t) \sim v\,t$, until it saturates at a value proportional to the subsystem size. Since an MPS with bond dimension $\chi$ can hold at most $S = \ln\chi$, a faithful simulation needs $\chi \sim e^{vt}$: the cost grows **exponentially with time**. This "entanglement barrier" limits MPS quench simulations to short times (or to slowly entangling dynamics), whereas ground states — which obey an *area law*, $S$ independent of the subsystem size in 1D — are where MPS excel.

The cell below runs two circuits through `CircuitMPS`: the GHZ circuit on 50 qubits (never needs $\chi > 2$) and the first-order Trotter circuit of Topic 4 for the TFIM from $|0\cdots0\rangle$, checked against exact evolution for $n = 8$ and then run on $n = 40$ qubits while recording the bond dimension and the half-chain entropy.

In [ ]:
# --- GHZ circuit on 50 qubits: an MPS never needs chi > 2 here ---
n = 50
circ = qtn.CircuitMPS(N=n, max_bond=8)
circ.h(0)
for j in range(n - 1):
    circ.cnot(j, j + 1)
print("GHZ-50: bond sizes", set(circ.psi.bond_sizes()), " amplitude('1'*50) =", circ.amplitude('1' * n), " <Z_0 Z_49> =", circ.local_expectation(Z & Z, (0, n - 1)).real)

# --- first-order Trotter circuit for the TFIM (open chain), from |0...0> ---
def trotter_circuit_mps(n, J, h, dt, n_steps, max_bond):
    circ = qtn.CircuitMPS(N=n, max_bond=max_bond, cutoff=1e-10)
    for _ in range(n_steps):
        for j in range(n - 1):
            circ.rzz(-2 * J * dt, j, j + 1)        # R_zz(theta) = e^{-i theta ZZ/2}  ->  e^{+i J dt Z Z}
        for j in range(n):
            circ.rx(-2 * h * dt, j)                # e^{+i h dt X}
    return circ

J, h, dt = 1.0, 1.2, 0.05
n = 8
H8 = ham_tfim(n, J, h)
psi0 = qu.computational_state('0' * n).ravel()
ts = np.arange(0, 41, 5) * dt
z_mps = [trotter_circuit_mps(n, J, h, dt, int(round(t / dt)), max_bond=16).local_expectation(Z, n // 2).real for t in ts]
z_exact = [np.vdot(v := expm_multiply(-1j * t * H8, psi0), qu.ikron(Z, [2] * n, n // 2) @ v).real for t in ts]
print("n=8, <Z_4>(t): max |CircuitMPS - exact| =", np.max(np.abs(np.array(z_mps) - np.array(z_exact))), " (first-order Trotter error)")

# --- n = 40: entanglement growth and bond dimension ---
n = 40
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for max_bond in [8, 32]:
    circ = qtn.CircuitMPS(N=n, max_bond=max_bond, cutoff=1e-10)
    S, chi_max_t, ts40 = [0.0], [1], [0.0]
    for step in range(1, 61):
        for j in range(n - 1):
            circ.rzz(-2 * J * dt, j, j + 1)
        for j in range(n):
            circ.rx(-2 * h * dt, j)
        S.append(entropy_cut(circ.psi, n // 2)); chi_max_t.append(max(circ.psi.bond_sizes())); ts40.append(step * dt)
    ax[0].plot(ts40, S, label=rf'$\chi_{{\max}} = {max_bond}$'); ax[1].plot(ts40, chi_max_t, label=rf'$\chi_{{\max}} = {max_bond}$')
ax[0].set_xlabel('t'); ax[0].set_ylabel('half-chain entropy S'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set_xlabel('t'); ax[1].set_ylabel('largest bond dimension'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**For further exploration.** `quimb` also has a dedicated TEBD class, `qtn.TEBD(psi0, H, dt)` with `H = qtn.ham_1d_ising(n, j=..., bx=...)` (a `LocalHam1D`), which builds the second- or fourth-order Trotter gates itself (`tebd.update_to(t, order=2)`, state in `tebd.pt`). Mind its conventions: it uses spin-$\tfrac12$ operators $S = \sigma/2$ and the sign $H = j\sum S^z S^z - b_x \sum S^x$, so our $H = -J\sum ZZ - h\sum X$ is `j=-4*J, bx=2*h`.

### Exercise 3: TEBD for a domain-wall quench

Consider the transverse-field Ising chain $H = -J\sum_{j=0}^{n-2} Z_j Z_{j+1} - h\sum_{j=0}^{n-1} X_j$ with $J = 1$, $h = 1.2$ (open boundaries), and the initial product state $|\psi_0\rangle = |{+}\cdots{+}\,{-}\cdots{-}\,{+}\cdots{+}\rangle$ of Section 1.1 (`domain_wall_mps`). It is an eigenstate of the field term; the $ZZ$ coupling makes the two "domain walls" in $\langle X_j\rangle$ melt and spread.

**1.** Write `trotter_step_(psi, dt, J, h, max_bond, info)` that applies one first-order step $U(\delta t)$ of Section 2.2 *in place* to an MPS, exactly as in the demo of Section 2.1: the gates $e^{+iJ\delta t\, Z_jZ_{j+1}}$ (`expm(1j * J * dt * np.kron(Z, Z))`) on all bonds $j = 0, \ldots, n-2$ from left to right with `psi.gate_(..., contract='swap+split', max_bond=max_bond, cutoff=1e-10, info=info)`, then $e^{+ih\delta t\,X_j}$ on every site with `contract=True`. Why is the left-to-right order convenient for the canonical form? Check one step for $n = 6$ against the dense vector: apply the same gates with `qu.ikron` to `domain_wall_mps(6).to_dense()` and compare.

**2.** Write `run_tebd(n, J, h, dt, n_steps, max_bond)` that starts from `domain_wall_mps(n)`, applies `n_steps` steps, and records after every step the profile $\langle X_j\rangle$ (`local_expectations`), the half-chain entropy (`entropy_cut(psi, n // 2)`) and the largest bond dimension. Run $n = 40$, $\chi_{\max} = 32$, $\delta t = 0.05$ up to $t = 4$ (keep the result as `xs40`, it is used in Exercise 4) and show $\langle X_j\rangle(t)$ as an image (`plt.imshow(xs40, aspect='auto', cmap='RdBu', vmin=-1, vmax=1, origin='lower', extent=[-0.5, n - 0.5, 0, t_max])`) next to $S(t)$ and the largest bond dimension versus $t$. Identify the light cone emanating from the two domain walls, and the time at which $S(t)$ reaches $\ln \chi_{\max}$ — beyond it the simulation is no longer converged.

In [ ]:
# --- Part 1: trotter_step_ ---
def trotter_step_(psi, dt, J, h, max_bond, info):
    pass


In [ ]:
# --- Part 2: run_tebd and the space-time plot for n = 40 ---
def run_tebd(n, J, h, dt, n_steps, max_bond):
    pass


### 2.3 The time-dependent variational principle (TDVP)

TEBD needs the propagator as a product of nearest-neighbour gates. The **TDVP** evolves the MPS directly with the Hamiltonian: it works for any $H$ that can be written as a **matrix product operator (MPO)** — including long-range interactions — and it conserves the norm and the energy exactly. The idea (Haegeman *et al.* 2011, 2016): the set of MPS with fixed bond dimensions is a manifold $\mathcal{M}_\chi$ inside the Hilbert space, and the Schrödinger equation is projected onto its **tangent space**,
$$
i\,\frac{d}{dt}|\psi(t)\rangle = P_{T}\, H\, |\psi(t)\rangle ,
$$
so that the state never leaves the manifold. The only approximation is the projection: if $\chi$ is large enough to hold the exact state, the evolution is exact (up to the time-stepping error).

**Matrix product operators.** An MPO is the operator analogue of an MPS: a chain of four-leg tensors $W_j$ with two physical legs (input $\sigma_j$, output $\sigma_j'$) and two virtual legs,
$$
H = \sum_{\sigma, \sigma'} W_0^{[\sigma_0'\sigma_0]} W_1^{[\sigma_1'\sigma_1]} \cdots W_{n-1}^{[\sigma_{n-1}'\sigma_{n-1}]}\; |\sigma'\rangle\langle\sigma| ,
\qquad
\begin{array}{c} |\quad\ \ |\qquad\quad\ \ | \\ [W_0]\text{--}[W_1]\text{--}\cdots\text{--}[W_{n-1}] \\ |\quad\ \ |\qquad\quad\ \ | \end{array}
$$
Think of $W_j$ as a small matrix whose entries are single-site operators. For the transverse-field Ising chain the bond dimension is $3$:
$$
W_j = \begin{pmatrix} \mathbb{1} & 0 & 0 \\ Z & 0 & 0 \\ -hX & -JZ & \mathbb{1} \end{pmatrix}, \qquad
W_0 = \begin{pmatrix} -hX & -JZ & \mathbb{1} \end{pmatrix} \ (\text{the last row}), \qquad
W_{n-1} = \begin{pmatrix} \mathbb{1} \\ Z \\ -hX \end{pmatrix} \ (\text{the first column}).
$$
Multiplying out the matrices, each product contains exactly one term $-J Z_j Z_{j+1}$ for every bond and $-hX_j$ for every site (the virtual index is a little *finite-state machine*: "nothing placed yet", "one $Z$ placed, need the partner", "term complete"). In `quimb`: `qtn.MPO_ham_ising(n, j=-4*J, bx=2*h)` (same spin-$\tfrac12$ conventions as `ham_1d_ising`).

**Effective Hamiltonians.** With the MPS in mixed canonical form, the Hamiltonian "seen" by the orthogonality center is obtained by contracting everything to the left and to the right of site $j$ into **environments**:
```
            ┌─[A0*]─[A1*]─ ... ─[Aj-1*]─       ─[Bj+1*]─ ... ─[Bn-1*]─┐
            │   |      |           |                 |            |    │
  L_{j-1} = │ ─[W0 ]─[W1 ]─ ... ─[Wj-1 ]─   R_{j+1} = ─[Wj+1 ]─ ... ─[Wn-1 ]─ │
            │   |      |           |                 |            |    │
            └─[A0 ]─[A1 ]─ ... ─[Aj-1 ]─       ─[Bj+1 ]─ ... ─[Bn-1 ]─┘
```
Each environment is a three-leg tensor (bra bond, MPO bond, ket bond) and is built iteratively, $L_j = L_{j-1} \cdot (A_j^*, W_j, A_j)$, at cost $O(\chi^3 w d + \chi^2 w^2 d^2)$ per site. The **one-site effective Hamiltonian** acting on the OC and the **zero-site effective Hamiltonian** acting on a bond matrix $S$ are
```
     ┌───────────┐                    ┌─────┐
     │  [  C  ]  │              ──────┤  S  ├──────
  L──┤     |     ├──R      and     L──┤     ├──R
     │  [  W  ]  │                    └─────┘
     └───────────┘
   H_eff^{(j)} C  =  L_{j-1} · W_j · R_{j+1} · C             K^{(j)} S  =  L_j · R_{j+1} · S
```
Both are Hermitian linear maps on vectors of length $2\chi^2$ and $\chi^2$ respectively; we never form them as matrices but only apply them (a few tensor contractions, $O(\chi^3 w d)$ each) and exponentiate them with the Krylov method of Topic 2 (`krylov_expm`).

**The algorithm (one-site TDVP, projector-splitting integrator).** The tangent-space projector of the MPS manifold decomposes into one term per site and one (negative) term per bond,
$$
P_T = \sum_{j=0}^{n-1} P^{A}_{[0,\,j-1]} \otimes \mathbb{1}_j \otimes P^{B}_{[j+1,\,n-1]} \;-\; \sum_{j=0}^{n-2} P^{A}_{[0,\,j]} \otimes P^{B}_{[j+1,\,n-1]} ,
$$
where $P^A$ ($P^B$) projects onto the space spanned by the left (right) isometric part of the chain. Splitting the exponential of this sum term by term (exactly like a Trotter decomposition, but of the *projector* rather than of $H$) turns one time step into a sweep in which each site is evolved **forward** with $H_{\rm eff}^{(j)}$ and each bond **backward** with $K^{(j)}$ — the backward step removes the double counting of the bond between two neighbouring site terms:

> **Input:** MPS in right-canonical form (OC at site 0), MPO $\{W_j\}$, step $\delta t$. Precompute $R_{n-1}, R_{n-2}, \ldots, R_1$.
>
> **Left-to-right sweep**, for $j = 0, 1, \ldots, n-1$:
> 1. $C_j \leftarrow e^{-i\,\delta t/2\, H_{\rm eff}^{(j)}}\, C_j$ (forward half step on site $j$),
> 2. if $j < n-1$: QR-decompose $C_j = A_j S$ ($A_j$ left-isometric); compute $L_j$ from $A_j$;
> 3. $\quad S \leftarrow e^{+i\,\delta t/2\, K^{(j)}}\, S$ (backward half step on the bond); $C_{j+1} \leftarrow S\, B_{j+1}$ (the OC moves to $j+1$).
>
> **Right-to-left sweep**, for $j = n-1, \ldots, 1, 0$:
> 1. $C_j \leftarrow e^{-i\,\delta t/2\, H_{\rm eff}^{(j)}}\, C_j$,
> 2. if $j > 0$: LQ-decompose $C_j = S\, B_j$ ($B_j$ right-isometric); compute $R_j$ from $B_j$;
> 3. $\quad S \leftarrow e^{+i\,\delta t/2\, K^{(j-1)}}\, S$; $C_{j-1} \leftarrow A_{j-1}\, S$.
>
> After the two sweeps the state is again right-canonical with the OC at site 0 and has advanced by $\delta t$.

Properties:
- Every substep is a unitary rotation generated by a Hermitian effective operator, so the **norm is conserved exactly**; for time-independent $H$ the **energy is conserved** as well (the symmetric sweep is a *symplectic* integrator on the manifold).
- The symmetric left–right / right–left sweep is **second order** in $\delta t$ (error $O(\delta t^3)$ per step); it has *no* Trotter error of the Hamiltonian itself.
- The one-site algorithm keeps all bond dimensions **fixed** — it cannot grow them. Starting from a product state one therefore pads the bonds to the desired $\chi$ (`psi.expand_bond_dimension(chi, rand_strength=1e-6)`, zeros plus a little noise) or uses the **two-site** variant (evolve $\Theta = C_j B_{j+1}$ forward, SVD-truncate, evolve the one-site tensor backward), which grows $\chi$ adaptively at the price of a truncation error.
- Cost per time step: $O(n\,\chi^3 w d)$ times the number of Krylov iterations, comparable with one TEBD step.

The cells below first build the MPO and check it, then implement the algorithm with `numpy` arrays extracted from the `quimb` MPS (`'lpr'` layout, MPO tensors in `quimb`'s `'lrud'` layout: left bond, right bond, output leg, input leg), and finally test it against exact evolution and against TEBD.

In [ ]:
n, J, h = 6, 1.0, 1.2
H_mpo = qtn.MPO_ham_ising(n, j=-4 * J, bx=2 * h)
print(H_mpo)
print("MPO == -J sum ZZ - h sum X :", np.allclose(np.asarray(H_mpo.to_dense()), ham_tfim(n, J, h, sparse=False)))
W = H_mpo.arrays[1]                    # (left bond, right bond, out, in) = 'lrud'
print("W_1 shape (l, r, out, in):", W.shape)
print("entries of W as 2x2 operators (quimb uses S = sigma/2):")
names = {tuple(I2.ravel()): '1', tuple(Z.ravel()): 'Z', tuple((Z / 2).ravel()): 'Z/2', tuple((-2 * J * Z).ravel()): '-2J Z', tuple((-h * X).ravel()): '-h X', tuple(np.zeros(4)): '0'}
for a in range(3):
    print("   [", "  ".join(f"{names.get(tuple(W[a, b].ravel()), '?'):>6s}" for b in range(3)), "]")
H_mpo.draw(color=[f'I{j}' for j in range(n)], show_inds='all', figsize=(7, 2))

In [ ]:
# ---------------- one-site TDVP with the MPS as a list of 'lpr' arrays ----------------
def mps_arrays(psi):
    '''Site tensors of a quimb MPS as arrays [alpha_{j-1}, sigma_j, alpha_j] (dummy bonds of size 1 at the ends).'''
    n, out = psi.L, []
    for j in range(n):
        inds = ([psi.bond(j - 1, j)] if j > 0 else []) + [psi.site_ind(j)] + ([psi.bond(j, j + 1)] if j < n - 1 else [])
        a = psi[j].transpose(*inds).data
        if j == 0: a = a[None]
        if j == n - 1: a = a[..., None]
        out.append(a)
    return out

def arrays_to_mps(As):
    return qtn.MatrixProductState([As[0][0]] + list(As[1:-1]) + [As[-1][..., 0]], shape='lpr')

def mpo_arrays(H):
    '''MPO tensors as arrays [w_{j-1}, w_j, sigma_out, sigma_in] (dummy bonds at the ends).'''
    Ws = list(H.arrays)
    Ws[0], Ws[-1] = Ws[0][None], Ws[-1][:, None]
    return Ws

# environments and effective Hamiltonians: each is a few tensordots, never a 2chi^2 x 2chi^2 matrix
def update_L(L, A, W):                                   # L[a', w, a] -> L'[b', w', b]
    tmp = np.tensordot(L, A, axes=([2], [0]))            # [a', w, s, b]
    tmp = np.tensordot(tmp, W, axes=([1, 2], [0, 3]))    # [a', b, w', s']
    return np.tensordot(A.conj(), tmp, axes=([0, 1], [0, 3])).transpose(0, 2, 1)   # [b', w', b]

def update_R(R, B, W):                                   # R[b', w', b] -> R'[a', w, a]
    tmp = np.tensordot(B, R, axes=([2], [2]))            # [a, s, b', w']
    tmp = np.tensordot(W, tmp, axes=([1, 3], [3, 1]))    # [w, s', a, b']
    return np.tensordot(B.conj(), tmp, axes=([1, 2], [1, 3]))   # [a', w, a]

def apply_heff1(L, W, R, C):                             # one-site effective Hamiltonian on C[a, s, b]
    tmp = np.tensordot(L, C, axes=([2], [0]))            # [a', w, s, b]
    tmp = np.tensordot(tmp, W, axes=([1, 2], [0, 3]))    # [a', b, w', s']
    return np.tensordot(tmp, R, axes=([1, 2], [2, 1]))   # [a', s', b']

def apply_keff0(L, R, S):                                # zero-site effective Hamiltonian on the bond matrix S[a, b]
    tmp = np.tensordot(L, S, axes=([2], [0]))            # [a', w, b]
    return np.tensordot(tmp, R, axes=([1, 2], [1, 2]))   # [a', b']

def krylov_expm(matvec, v, z, m_max=30, tol=1e-10):
    '''exp(z H) v for a Hermitian H given by its action `matvec` (Topic 2, with a convergence test on the Krylov dimension).'''
    shape, v = v.shape, v.ravel()
    nrm = np.linalg.norm(v)
    Q, alphas, betas, prev = [v / nrm], [], [], None
    for k in range(m_max):
        w = matvec(Q[k].reshape(shape)).ravel()
        a = np.vdot(Q[k], w).real
        w = w - a * Q[k] - (betas[-1] * Q[k - 1] if k > 0 else 0)
        for q in Q:                                      # full re-orthogonalization (cheap: the vectors are short)
            w = w - np.vdot(q, w) * q
        alphas.append(a); b = np.linalg.norm(w)
        T = np.diag(alphas) + np.diag(betas, 1) + np.diag(betas, -1)
        e = expm(z * T)[:, 0]                            # exp(zT) applied to e_1
        if b < tol or (prev is not None and np.linalg.norm(e[:-1] - prev) < tol and abs(e[-1]) < tol):
            break
        prev = e; betas.append(b); Q.append(w / b)
    return (nrm * (np.array(Q[:len(e)]).T @ e)).reshape(shape)

class TDVP1:
    '''One-site TDVP (projector-splitting integrator) for an MPS and an MPO Hamiltonian.'''
    def __init__(self, psi, H):
        psi = psi.copy(); psi.canonize(0)                # right-canonical, OC at site 0
        self.As, self.Ws, self.n = mps_arrays(psi), mpo_arrays(H), psi.L
        self.Ls, self.Rs = [None] * self.n, [None] * self.n
        R = np.ones((1, 1, 1))
        for j in range(self.n - 1, 0, -1):               # right environments from the B's
            R = update_R(R, self.As[j], self.Ws[j]); self.Rs[j] = R

    def L(self, j): return self.Ls[j] if j >= 0 else np.ones((1, 1, 1))
    def R(self, j): return self.Rs[j] if j < self.n else np.ones((1, 1, 1))

    def step(self, dt):
        n, As, Ws = self.n, self.As, self.Ws
        for j in range(n):                                                             # ---- left to right ----
            C = krylov_expm(lambda v: apply_heff1(self.L(j - 1), Ws[j], self.R(j + 1), v), As[j], -1j * dt / 2)
            if j == n - 1:
                As[j] = C; break
            l, p, r = C.shape
            Q, S = qr(C.reshape(l * p, r), mode='economic')                            # C_j = A_j S
            As[j] = Q.reshape(l, p, -1); self.Ls[j] = update_L(self.L(j - 1), As[j], Ws[j])
            S = krylov_expm(lambda v: apply_keff0(self.L(j), self.R(j + 1), v), S, +1j * dt / 2)
            As[j + 1] = np.tensordot(S, As[j + 1], axes=([1], [0]))                    # C_{j+1} = S B_{j+1}
        for j in range(n - 1, -1, -1):                                                 # ---- right to left ----
            C = krylov_expm(lambda v: apply_heff1(self.L(j - 1), Ws[j], self.R(j + 1), v), As[j], -1j * dt / 2)
            if j == 0:
                As[j] = C; break
            l, p, r = C.shape
            Q, S = qr(C.reshape(l, p * r).T, mode='economic')                          # C_j = S B_j  (LQ via QR of the transpose)
            As[j] = Q.T.reshape(-1, p, r); S = S.T
            self.Rs[j] = update_R(self.R(j + 1), As[j], Ws[j])
            S = krylov_expm(lambda v: apply_keff0(self.L(j - 1), self.R(j), v), S, +1j * dt / 2)
            As[j - 1] = np.tensordot(As[j - 1], S, axes=([2], [0]))                    # C_{j-1} = A_{j-1} S

    def mps(self): return arrays_to_mps(self.As)
    def energy(self):                                   # <H> from the OC at site 0 and its environments
        C = self.As[0]
        return np.vdot(C, apply_heff1(self.L(-1), self.Ws[0], self.R(1), C)).real
    def norm(self): return np.linalg.norm(self.As[0])

def expect_mpo(psi, H):
    '''<psi|H|psi> / <psi|psi> for any MPS psi and MPO H, by contracting the environments from the right.'''
    As, Ws = mps_arrays(psi), mpo_arrays(H)
    R = np.ones((1, 1, 1))
    for j in range(len(As) - 1, -1, -1):
        R = update_R(R, As[j], Ws[j])
    return R.item().real / (psi.H @ psi).real

In [ ]:
# --- test 1: n = 10, exact representability (chi = 32) -> TDVP should agree with exact evolution ---
n, J, h, dt, t_max = 10, 1.0, 1.2, 0.05, 2.0
n_steps = int(round(t_max / dt))
H_mpo = qtn.MPO_ham_ising(n, j=-4 * J, bx=2 * h)
H10 = ham_tfim(n, J, h)
Xd = [qu.ikron(X, [2] * n, j) for j in range(n)]

psi0 = domain_wall_mps(n)
v = np.asarray(psi0.to_dense()).ravel()
xs_exact = [[np.vdot(v, Xd[j] @ v).real for j in range(n)]]
for _ in range(n_steps):
    v = expm_multiply(-1j * dt * H10, v); xs_exact.append([np.vdot(v, Xd[j] @ v).real for j in range(n)])
xs_exact = np.array(xs_exact)
ts = np.arange(n_steps + 1) * dt

fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
for chi in [4, 8, 32]:
    psi = domain_wall_mps(n); psi.expand_bond_dimension(chi, rand_strength=1e-6)     # pad the bonds: one-site TDVP cannot grow them
    tdvp = TDVP1(psi, H_mpo)
    xs, Es = [local_expectations(tdvp.mps(), X)], [tdvp.energy()]
    for _ in range(n_steps):
        tdvp.step(dt); xs.append(local_expectations(tdvp.mps(), X)); Es.append(tdvp.energy())
    ax[0].semilogy(ts, np.maximum(np.abs(np.array(xs) - xs_exact).max(axis=1), 1e-16), label=rf'$\chi = {chi}$')
    ax[1].semilogy(ts, np.maximum(np.abs(np.array(Es) - Es[0]), 1e-16), label=rf'$\chi = {chi}$')
    print(f"chi={chi:2d}: norm = {tdvp.norm():.12f}, energy drift = {np.max(np.abs(np.array(Es) - Es[0])):.1e}, E = {Es[0]:.4f} (exact: {np.vdot(v, H10 @ v).real:.4f})")
ax[0].set_xlabel('t'); ax[0].set_ylabel(r'$\max_j|\langle X_j\rangle - $ exact$|$'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set_xlabel('t'); ax[1].set_ylabel(r'$|E(t) - E(0)|$'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- test 2: the n = 40 domain-wall quench with TDVP, chi = 32 (compare with the TEBD picture of Exercise 3) ---
import time
n, J, h, dt, t_max = 40, 1.0, 1.2, 0.05, 4.0
n_steps = int(round(t_max / dt))
H_mpo = qtn.MPO_ham_ising(n, j=-4 * J, bx=2 * h)
psi = domain_wall_mps(n); psi.expand_bond_dimension(32, rand_strength=1e-6)
tdvp = TDVP1(psi, H_mpo)
xs_tdvp, S_tdvp, E_tdvp = [local_expectations(tdvp.mps(), X)], [0.0], [tdvp.energy()]
t0 = time.time()
for _ in range(n_steps):
    tdvp.step(dt)
    m = tdvp.mps(); xs_tdvp.append(local_expectations(m, X)); S_tdvp.append(entropy_cut(m, n // 2)); E_tdvp.append(tdvp.energy())
xs_tdvp = np.array(xs_tdvp); ts40 = np.arange(n_steps + 1) * dt
print(f"{n_steps} TDVP steps in {time.time() - t0:.1f} s;  energy drift {np.max(np.abs(np.array(E_tdvp) - E_tdvp[0])):.1e};  norm {tdvp.norm():.10f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
im = ax[0].imshow(xs_tdvp, aspect='auto', cmap='RdBu', vmin=-1, vmax=1, origin='lower', extent=[-0.5, n - 0.5, 0, t_max])
ax[0].set_xlabel('site j'); ax[0].set_ylabel('t'); ax[0].set_title(r'$\langle X_j\rangle(t)$, TDVP $\chi = 32$'); plt.colorbar(im, ax=ax[0])
ax[1].plot(ts40, S_tdvp); ax[1].axhline(np.log(32), color='gray', ls=':', label=r'$\ln 32$')
ax[1].set_xlabel('t'); ax[1].set_ylabel('half-chain entropy'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Exercise 4: TDVP versus TEBD

Compare the TDVP run of the demo above (`xs_tdvp`, $n = 40$, $\chi = 32$) with your TEBD run of Exercise 3 (`xs40`, $\chi_{\max} = 32$, same $\delta t$): plot $\max_j |\langle X_j\rangle_{\rm TDVP} - \langle X_j\rangle_{\rm TEBD}|$ versus $t$ on a log scale. Up to what time do the two agree, and which sources of error does each method have? Which quantities does TDVP conserve exactly that TEBD does not (look at `E_tdvp` from the demo, and think about what truncation and Trotterization do to $\langle H\rangle$)?

In [ ]:
# --- TDVP vs TEBD on the same quench ---


---

### References

- U. Schollwöck, *The density-matrix renormalization group in the age of matrix product states*, Ann. Phys. **326**, 96 (2011) — the standard reference for MPS, canonical forms, MPOs and TEBD.
- G. Vidal, *Efficient classical simulation of slightly entangled quantum computations*, Phys. Rev. Lett. **91**, 147902 (2003) — the MPS circuit simulator / TEBD.
- J. Haegeman, C. Lubich, I. Oseledets, B. Vandereycken, F. Verstraete, *Unifying time evolution and optimization with matrix product states*, Phys. Rev. B **94**, 165116 (2016) — the projector-splitting TDVP of Section 2.3.
- S. Paeckel *et al.*, *Time-evolution methods for matrix-product states*, Ann. Phys. **411**, 167998 (2019) — a review comparing TEBD, TDVP and other methods.
- `quimb` documentation: [MPS basics](https://quimb.readthedocs.io/en/latest/tensor-1d.html), [circuit simulation](https://quimb.readthedocs.io/en/latest/tensor-circuit.html).